In [11]:
%pip uninstall -y tensorflow tensorflow-gpu tensorflow-probability tf-keras onnx onnx-tf protobuf tensorflow-hub dopamine-rl


Found existing installation: tensorflow 2.12.0
Uninstalling tensorflow-2.12.0:
  Successfully uninstalled tensorflow-2.12.0
Found existing installation: tensorflow-probability 0.19.0
Uninstalling tensorflow-probability-0.19.0:
  Successfully uninstalled tensorflow-probability-0.19.0
Found existing installation: onnx 1.13.1
Uninstalling onnx-1.13.1:
  Successfully uninstalled onnx-1.13.1
Found existing installation: onnx-tf 1.10.0
Uninstalling onnx-tf-1.10.0:
  Successfully uninstalled onnx-tf-1.10.0
Found existing installation: protobuf 3.20.3
Uninstalling protobuf-3.20.3:
  Successfully uninstalled protobuf-3.20.3
Note: you may need to restart the kernel to use updated packages.


You can safely remove it manually.


In [12]:
%pip install protobuf==3.20.3
%pip install tensorflow==2.12.0
%pip install numpy>=1.24.0
%pip install keras==2.12.0
%pip install onnx==1.13.0
%pip install onnx-tf==1.9.0
%pip install tensorflow-probability==0.19.0

  Using cached protobuf-3.20.3-cp310-cp310-win_amd64.whl.metadata (698 bytes)
Using cached protobuf-3.20.3-cp310-cp310-win_amd64.whl (904 kB)
Note: you may need to restart the kernel to use updated packages.
  Using cached tensorflow-2.12.0-cp310-cp310-win_amd64.whl.metadata (2.5 kB)
Using cached tensorflow-2.12.0-cp310-cp310-win_amd64.whl (1.9 kB)
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
  Using cached onnx-1.13.0-cp310-cp310-win_amd64.whl.metadata (14 kB)
Using cached onnx-1.13.0-cp310-cp310-win_amd64.whl (12.2 MB)
Note: you may need to restart the kernel to use updated packages.
  Using cached onnx_tf-1.9.0-py3-none-any.whl.metadata (508 bytes)
Using cached onnx_tf-1.9.0-py3-none-any.whl (222 kB)
Note: you may need to restart the kernel to use updated packages.
  Using cached tensorflow_probability-0.19.0-py2.py3-none-any.whl.me

In [2]:
import os
import onnx
import numpy as np
import tensorflow as tf
from onnx_tf.backend import prepare
import gc

c:\Users\Asus TUF -PC\miniconda3\envs\torch_env\lib\site-packages\tensorflow_addons\utils\tfa_eol_msg.py:23: UserWarning: 

TensorFlow Addons (TFA) has ended development and introduction of new features.
TFA has entered a minimal maintenance and release mode until a planned end of life in May 2024.
Please modify downstream libraries to take dependencies from other repositories in our TensorFlow community (e.g. Keras, Keras-CV, and Keras-NLP). 

For more information see: https://github.com/tensorflow/addons/issues/2807 

  warnings.warn(


In [5]:
def convert_mobilenet_to_tflite(onnx_path, tflite_output_path):
    # Clear any existing TensorFlow session
    tf.keras.backend.clear_session()
    gc.collect()

    try:
        # Load ONNX model
        print("Loading ONNX model...")
        onnx_model = onnx.load(onnx_path)

        # Convert ONNX to TensorFlow
        print("Converting to TensorFlow...")
        tf_rep = prepare(onnx_model)

        # Export TF model
        tf_model_dir = "temp_tf_model"
        print(f"Exporting TF model to {tf_model_dir}...")
        tf_rep.export_graph(tf_model_dir)

        # Convert to TFLite
        print("Converting to TFLite...")
        converter = tf.lite.TFLiteConverter.from_saved_model(tf_model_dir)

        # Basic configurations
        converter.target_spec.supported_ops = [
            tf.lite.OpsSet.TFLITE_BUILTINS,
            tf.lite.OpsSet.SELECT_TF_OPS
        ]
        converter.allow_custom_ops = True

        # Disable optimizations temporarily to ensure conversion works
        # converter.optimizations = [tf.lite.Optimize.DEFAULT]

        tflite_model = converter.convert()
        with open(tflite_output_path, "wb") as f:
            f.write(tflite_model)
        print(f"Successfully converted and saved to {tflite_output_path}")

        # Print model size
        print(f"Model size: {len(tflite_model) / (1024*1024):.2f} MB")
        return True

    except Exception as e:
        print(f"Conversion failed with error: {str(e)}")
        return False

# Convert the model
convert_mobilenet_to_tflite("leaf_classifier_81.onnx", "model.tflite")

Loading ONNX model...
Converting to TensorFlow...
Exporting TF model to temp_tf_model...


INFO:tensorflow:Assets written to: temp_tf_model\assets


INFO:tensorflow:Assets written to: temp_tf_model\assets


Converting to TFLite...
Successfully converted and saved to model.tflite
Model size: 13.75 MB


True